# Dijet data distributions

Beam-orientation comparisons for configurable reconstructed-dijet selections. For the unflipped-Lab, flipped-Lab, and CM frames, plot common-scale Pb-going, p-going, and combined 2D maps, normalized pTave spectra with ratios to combined, and eta projections with ratios to combined. In the CM frame, also compare unnormalized forward/backward ratios and their ratios to the combined result.

All four triggers use the shared data-output resolver, which maps each configured selection to the combined, Pb-going, and p-going merged filenames.


## Environment and imports

This notebook locates the repository dynamically and imports PyROOT from the
active project environment. Start Jupyter from the repository root with
`py-env/bin/python -m jupyter notebook`; no machine-specific ROOT paths are
added at runtime.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from dataclasses import replace
import os

import sys

# Locate the repository without relying on a machine-specific absolute path.
PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "CMakeLists.txt").is_file()
        and (candidate / "hist_analysis").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(
        "Cannot locate the jetAnalysis repository. Start Jupyter from its root."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hist_analysis.python.notebook_setup import load_root

# Batch mode keeps plots reproducible and sends them to notebook/output files.
ROOT = load_root(batch=True)

from hist_analysis.python.notebook_setup import load_root

# Batch mode keeps plots reproducible and sends them to notebook/output files.
ROOT = load_root(batch=True)

from hist_analysis.config.files import BASE_DIR
from hist_analysis.python.data_distributions import (
    draw_orientation_comparisons,
)
from hist_analysis.python.histogram_io import resolve_data_file
from hist_analysis.python.root_style import (
    DEFAULT_PLOT_STYLE, save_canvas, set_1d_style, set_pad_style,
)


In [ ]:
ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetPalette(ROOT.kBird)
ROOT.TH1.AddDirectory(False)

In [ ]:
DATA_DIR = Path(os.environ.get('PPB_DATA_DIR', BASE_DIR / 'exp'))
SELECTION = 'jetId'  # jetId, trkMax, or noSel
if SELECTION not in {'jetId', 'trkMax', 'noSel'}: raise ValueError(f'Unsupported SELECTION={SELECTION!r}')
OUTPUT_DIR = Path(os.environ.get('DATA_DIJET_OUTPUT_DIR', PROJECT_ROOT / 'hist_analysis/output/data_dijet_distributions')) / SELECTION
TRIGGERS = ('MinimumBias', 'Jet60', 'Jet80', 'Jet100')
DIRECTION_FILES = {trigger: {
    label: resolve_data_file(DATA_DIR, trigger, direction, SELECTION)
    for label, direction in (
        ('Pb-going', 'Pbgoing'), ('p-going', 'pgoing'), ('combined', 'combined'),
    )
} for trigger in TRIGGERS}
PTAVE_BINS = {
    'MinimumBias': [(50, 60), (60, 70), (70, 80)],
    # 'MinimumBias': [(60, 80), (80, 100), (100, 120), (120, 180)],
    'Jet60': [(80, 90), (90, 100)],
    # 'Jet60': [(80, 100), (100, 120), (120, 180), (180, 250)],
    'Jet80': [(100, 110), (110, 120)],
    # 'Jet80': [(100, 120), (120, 180), (180, 250), (300, 500)],
    'Jet100': [(120, 130), (130, 140), (140, 150), (150, 160), (160, 180), (180, 200), (200, 250), (250, 300), (300, 500)],
    # 'Jet100': [(120, 180), (180, 250), (300, 500)],
}
FRAMES = {
    'lab_unflipped': ('Lab unflipped', 'hRecoDijetPtEtaLabUnflipped', '#eta_{Lab,unflipped}^{dijet}'),
    'lab': ('Lab flipped', 'hRecoDijetPtEtaLab', '#eta_{Lab}^{dijet}'),
    'cm': ('CM', 'hRecoDijetPtEtaCM', '#eta_{CM}^{dijet}'),
}
REBIN_PT = 1; REBIN_ETA = 2; PT_DISPLAY_RANGE = (40.0, 500.0)
ETA_CUT = 1.9
ETA_DISPLAY_RANGE = (-ETA_CUT - 0.1, ETA_CUT + 0.1)
FB_DISPLAY_RANGE = (0.0, ETA_CUT + 0.1)
ETA_NORMALIZATION = 'bin_width'  # true 1/N dN/deta normalization
PT_ORIENTATION_NORMALIZATION_RANGE = (110.0, 130.0)
ORIENTATION_RATIO_RANGE = (0.75, 1.25)
FB_RANGE = (0.5, 1.5); FB_ORIENTATION_RATIO_RANGE = (0.75, 1.25)
SAVE_PNG = False; DRAW_GRID = True
PLOT_STYLE = replace(DEFAULT_PLOT_STYLE, annotation_text_size=0.028, legend_text_size=0.028)
SQUARE_PLOT_STYLE = replace(PLOT_STYLE, canvas_width=800, canvas_height=800)
missing = [str(path) for files in DIRECTION_FILES.values() for path in files.values() if not path.exists()]
if missing: raise FileNotFoundError('Missing ROOT files:\n' + '\n'.join(sorted(set(missing))))


## Pb-going and p-going overlays and ratios to combined

In [ ]:
mb_orientation_results = draw_orientation_comparisons(
    'MinimumBias', DIRECTION_FILES['MinimumBias'], FRAMES, PTAVE_BINS['MinimumBias'],
    jet_kind='dijet', output_dir=OUTPUT_DIR, rebin_eta=REBIN_ETA, rebin_pt=REBIN_PT,
    pt_display_range=PT_DISPLAY_RANGE, pt_eta_range=None,
    pt_normalization_range=PT_ORIENTATION_NORMALIZATION_RANGE,
    selection=SELECTION,
    ratio_range=ORIENTATION_RATIO_RANGE, save_png=SAVE_PNG,
    grid=DRAW_GRID, style=PLOT_STYLE, eta_normalization=ETA_NORMALIZATION,
    eta_display_range=ETA_DISPLAY_RANGE,
    fb_y_range=FB_RANGE, fb_x_range=FB_DISPLAY_RANGE,
    fb_ratio_range=FB_ORIENTATION_RATIO_RANGE,
)

In [ ]:
jet60_orientation_results = draw_orientation_comparisons(
    'Jet60', DIRECTION_FILES['Jet60'], FRAMES, PTAVE_BINS['Jet60'],
    jet_kind='dijet', output_dir=OUTPUT_DIR, rebin_eta=REBIN_ETA, rebin_pt=REBIN_PT,
    pt_display_range=PT_DISPLAY_RANGE, pt_eta_range=None,
    pt_normalization_range=PT_ORIENTATION_NORMALIZATION_RANGE,
    selection=SELECTION,
    ratio_range=ORIENTATION_RATIO_RANGE, save_png=SAVE_PNG,
    grid=DRAW_GRID, style=PLOT_STYLE, eta_normalization=ETA_NORMALIZATION,
    eta_display_range=ETA_DISPLAY_RANGE,
    fb_y_range=FB_RANGE, fb_x_range=FB_DISPLAY_RANGE,
    fb_ratio_range=FB_ORIENTATION_RATIO_RANGE,
)

In [ ]:
jet80_orientation_results = draw_orientation_comparisons(
    'Jet80', DIRECTION_FILES['Jet80'], FRAMES, PTAVE_BINS['Jet80'],
    jet_kind='dijet', output_dir=OUTPUT_DIR, rebin_eta=REBIN_ETA, rebin_pt=REBIN_PT,
    pt_display_range=PT_DISPLAY_RANGE, pt_eta_range=None,
    pt_normalization_range=PT_ORIENTATION_NORMALIZATION_RANGE,
    selection=SELECTION,
    ratio_range=ORIENTATION_RATIO_RANGE, save_png=SAVE_PNG,
    grid=DRAW_GRID, style=PLOT_STYLE, eta_normalization=ETA_NORMALIZATION,
    eta_display_range=ETA_DISPLAY_RANGE,
    fb_y_range=FB_RANGE, fb_x_range=FB_DISPLAY_RANGE,
    fb_ratio_range=FB_ORIENTATION_RATIO_RANGE,
)

In [ ]:
jet100_orientation_results = draw_orientation_comparisons(
    'Jet100', DIRECTION_FILES['Jet100'], FRAMES, PTAVE_BINS['Jet100'],
    jet_kind='dijet', output_dir=OUTPUT_DIR, rebin_eta=REBIN_ETA, rebin_pt=REBIN_PT,
    pt_display_range=PT_DISPLAY_RANGE, pt_eta_range=None,
    pt_normalization_range=PT_ORIENTATION_NORMALIZATION_RANGE,
    selection=SELECTION,
    ratio_range=ORIENTATION_RATIO_RANGE, save_png=SAVE_PNG,
    grid=DRAW_GRID, style=PLOT_STYLE, eta_normalization=ETA_NORMALIZATION,
    eta_display_range=ETA_DISPLAY_RANGE,
    fb_y_range=FB_RANGE, fb_x_range=FB_DISPLAY_RANGE,
    fb_ratio_range=FB_ORIENTATION_RATIO_RANGE,
)

## Combined-orientation CM 4x4 summaries

Collect all 16 selected pTave intervals into two square 8x8-style figures: a 4x4 normalized CM-pseudorapidity summary and a separate 4x4 Forward/Backward summary. Each pad contains one histogram and no legend.

In [ ]:
orientation_results = {
    'MinimumBias': mb_orientation_results,
    'Jet60': jet60_orientation_results,
    'Jet80': jet80_orientation_results,
    'Jet100': jet100_orientation_results,
}
combined_intervals = []
for trigger, results in orientation_results.items():
    for ptave_range in PTAVE_BINS[trigger]:
        low, high = ptave_range
        interval_result = results[('cm', ptave_range)]
        combined_intervals.append({
            'ptave_range': ptave_range,
            'eta': interval_result['histograms']['combined'],
            'fb': interval_result['forward_backward']['combined'],
        })

if len(combined_intervals) != 16:
    raise ValueError(f'Expected 16 selected pTave intervals, got {len(combined_intervals)}')
for entry in combined_intervals:
    area = entry['eta'].Integral('width')
    if abs(area - 1.0) >= 1e-9:
        raise AssertionError(f'CM eta density has area {area}, expected 1')

def draw_combined_summary(observable):
    is_fb = observable == 'fb'
    canvas = ROOT.TCanvas(
        f'c_combined_cm_{observable}_4x4', '',
        SQUARE_PLOT_STYLE.canvas_width, SQUARE_PLOT_STYLE.canvas_height,
    )
    canvas.Divide(4, 4, 0.001, 0.001)
    retained = []
    for pad_index, entry in enumerate(combined_intervals, 1):
        pad = canvas.cd(pad_index)
        set_pad_style(pad, grid_x=DRAW_GRID, grid_y=DRAW_GRID, style=SQUARE_PLOT_STYLE)
        histogram = entry[observable]
        set_1d_style(histogram, 2, style=SQUARE_PLOT_STYLE)
        histogram.SetTitle('')
        histogram.GetXaxis().SetTitle('|#eta_{CM}^{dijet}|' if is_fb else '#eta_{CM}^{dijet}')
        histogram.GetYaxis().SetTitle('Forward / Backward' if is_fb else '1/N dN/d#eta_{CM}^{dijet}')
        histogram.GetXaxis().SetRangeUser(*(FB_DISPLAY_RANGE if is_fb else ETA_DISPLAY_RANGE))
        if is_fb:
            histogram.SetMinimum(FB_RANGE[0])
            histogram.SetMaximum(FB_RANGE[1])
        else:
            histogram.SetMinimum(0.0)
            histogram.SetMaximum(1.25 * histogram.GetMaximum())
        histogram.Draw('E1')
        if is_fb:
            reference = ROOT.TLine(FB_DISPLAY_RANGE[0], 1.0, FB_DISPLAY_RANGE[1], 1.0)
            reference.SetLineStyle(2)
            reference.Draw('SAME')
            retained.append(reference)
        low, high = entry['ptave_range']
        text = ROOT.TLatex()
        text.SetNDC(True)
        text.SetTextFont(42)
        text.SetTextSize(0.065)
        text.DrawLatex(0.22, 0.84, f'{low:g} #leq p_{{T}}^{{ave}} < {high:g} GeV')
        retained.extend((histogram, text))
        pad.Modified()
    canvas.Modified()
    canvas.Update()
    output = OUTPUT_DIR / f'combined_cm_dijet_{observable}_4x4.pdf'
    save_canvas(canvas, output, save_png=SAVE_PNG)
    canvas._combined_summary_objects = retained
    return canvas

combined_eta_summary_canvas = draw_combined_summary('eta')
combined_fb_summary_canvas = draw_combined_summary('fb')
display(combined_eta_summary_canvas)
display(combined_fb_summary_canvas)